# Toxic Comment Multi-Label Classification

Run **Kernel → Restart & Run All** to execute the full pipeline end-to-end.

Pipeline:
1. Setup & Import
2. Load Dataset
3. Traditional Pipeline (TF-IDF + Logistic Regression + Optuna)
4. Deep Learning Pipeline (DistilRoBERTa embeddings + PyTorch)

In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

# Standard libraries
import pandas as pd
import numpy as np
import time

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import (
    classification_report, f1_score, precision_score, recall_score,
    roc_auc_score, accuracy_score, hamming_loss, precision_recall_fscore_support,
)

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

# ── Module imports ──────────────────────────────────────────────────────────
from modules.preprocess_data import normalize_text, clean_text_dl
from modules.traditional_pipeline import fit_tfidf, run_optuna_training
from modules.deep_learning_pipeline import (
    extract_sentence_embeddings,
    DeepLogisticRegression,
    train_dl_model,
    evaluate_dl_model,
)

print('Environment setup complete!')

Environment setup complete!


# Load Dataset

In [2]:
train_df = pd.read_csv('../data/raw/train.csv')
test_df  = pd.read_csv('../data/raw/test.csv')

print('Train:', train_df.shape)
print('Test: ', test_df.shape)

label_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
X_train = train_df['comment_text'].values
X_test  = test_df['comment_text'].values
y_train = train_df[label_cols]

print('Number of Null values:', train_df.isnull().sum().values.sum())

counts = train_df[label_cols].sum(axis=1).value_counts().sort_index()
print('Label count distribution (%):')
print((counts / len(train_df) * 100).round(2))

multi = (train_df[label_cols].sum(axis=1) > 1).mean()
print('Fraction of rows with multiple labels:', round(multi, 4))

train_df['is_toxic'] = train_df[label_cols].max(axis=1)
toxic_df = train_df[train_df['is_toxic'] == 1]
clean_df  = train_df[train_df['is_toxic'] == 0]
print('Toxic:', toxic_df.shape, '| Clean:', clean_df.shape)

Train: (159571, 8)
Test:  (153164, 2)
Number of Null values: 0
Label count distribution (%):
0    89.83
1     3.99
2     2.18
3     2.64
4     1.10
5     0.24
6     0.02
Name: count, dtype: float64
Fraction of rows with multiple labels: 0.0618
Toxic: (16225, 9) | Clean: (143346, 9)


---
# Traditional Pipeline
## Step 1 — Text Preprocessing & Tokenization

Uses `normalize_text` from `modules/preprocess_data.py`:
lower-case → unidecode → contractions → HTML-unescape → strip HTML/URLs/IPs → remove punctuation → stopword removal → Porter stemming.

In [ ]:
import pickle, os, subprocess, sys

NORMALIZED_PATH = "../features/normalized_texts.pkl"
SCRIPT_PATH     = os.path.abspath("../preprocess_and_save.py")

if not os.path.exists(NORMALIZED_PATH):
    print("Pre-processed texts not found. Running preprocess_and_save.py ...")
    print("(This will take a few minutes. Progress is shown below.)\n")
    result = subprocess.run(
        [sys.executable, SCRIPT_PATH],
        cwd=os.path.abspath(".."),
    )
    if result.returncode != 0:
        raise RuntimeError(
            "preprocess_and_save.py failed (see output above).\n"
            "If it crashed mid-way, just re-run this cell — it will resume from the last checkpoint."
        )
else:
    print("Pre-processed texts found on disk. Loading...")

with open(NORMALIZED_PATH, "rb") as f:
    _data = pickle.load(f)

X_normalize      = _data["X_normalize"]
X_test_normalize = _data["X_test_normalize"]
X_clean      = [clean_text_dl(text) for text in X_train]
X_test_clean = [clean_text_dl(text) for text in X_test]

print(f"Loaded: {len(X_normalize):,} train | {len(X_test_normalize):,} test")
print("\nSample record after DL cleaning:")
print(X_clean[0])


1. Xử lý tập Train:


Batch 1/16:   0%|          | 0/10000 [00:00<?, ?it/s]

  Batch 1 saved to disk.


Batch 2/16:   0%|          | 0/10000 [00:00<?, ?it/s]

  Batch 2 saved to disk.


...

  Batch 10 saved to disk.


Batch 11/16:   0%|          | 0/10000 [00:00<?, ?it/s]

  Batch 11 saved to disk.


Batch 12/16:   0%|          | 0/10000 [00:00<?, ?it/s]

: 

## Step 2 — Feature Extraction (TF-IDF)

Uses `fit_tfidf` from `modules/traditional_pipeline.py`.  
Saves sparse matrices to `../features/tfidf_*.h5`.

In [ ]:
X_train_tfidf, X_test_tfidf, tfidf = fit_tfidf(
    X_normalize,
    X_test_normalize,
    max_features=10000,
    features_dir='../features',
)

## Step 3 — End-to-End Training (Optuna + Threshold Tuning)

Uses `run_optuna_training` from `modules/traditional_pipeline.py`.

- Multilabel-stratified split (80/20 → tune/report)
- Optuna hyperparameter search (3-fold CV)
- Per-label threshold tuning
- Evaluation on holdout split
- Retrain on 100 % → save `../models/test_pred_proba.csv`, `best_thresholds.csv`

> Set `n_trials` lower (e.g. 5) for a quick test run.

In [ ]:
trad_model, best_thresholds = run_optuna_training(
    X_train_tfidf,
    X_test_tfidf,
    y_all=train_df[label_cols],
    test_df=test_df,
    label_cols=label_cols,
    models_dir='../models',
    n_trials=20,
)

---
# Deep Learning Pipeline
## Step 1 — Text Preprocessing

Uses `clean_text_dl` from `modules/preprocess_data.py`:
HTML-unescape → strip HTML/URLs → unidecode → contractions → lower-case → remove special chars.

In [ ]:
print('Starting text preprocessing for Deep Learning Pipeline...')

X_clean      = [clean_text_dl(text) for text in X_train]
X_test_clean = [clean_text_dl(text) for text in X_test]

print('Sample record after DL cleaning:')
print(X_clean[0])

## Step 2 — Feature Extraction (DistilRoBERTa Embeddings)

Uses `extract_sentence_embeddings` from `modules/deep_learning_pipeline.py`.  
Model: `all-distilroberta-v1` → 768-dim vectors.  
Saves to `../features/X_train_embeddings.npy` and `X_test_embeddings.npy`.

In [ ]:
X_train_embds, X_test_embds = extract_sentence_embeddings(
    X_clean,
    X_test_clean,
    features_dir='../features',
    model_name='all-distilroberta-v1',
)

## Step 3 — PyTorch Training

Uses `train_dl_model` from `modules/deep_learning_pipeline.py`.  
Architecture: single linear layer (`DeepLogisticRegression`) with `BCEWithLogitsLoss` + smoothed class weights.  
Saves model to `../models/pytorch_logreg_smoothed.pth`.

In [ ]:
y_train_dl = train_df[label_cols].values[:X_train_embds.shape[0]]

dl_model, device = train_dl_model(
    X_train_dl=X_train_embds,
    y_train_dl=y_train_dl,
    models_dir='../models',
    epochs=30,
    batch_size=1024,
    lr=0.01,
)

## Step 4 — Evaluation

Uses `evaluate_dl_model` from `modules/deep_learning_pipeline.py`.  
Merges predictions with ground-truth labels, removes Kaggle un-scored rows (-1),  
then calls `evaluation_metrics.export_metrics` for consistent reporting.  
Outputs saved to `../reports/evaluation_output_dl/`.

In [ ]:
results_dl = evaluate_dl_model(
    dl_model=dl_model,
    device=device,
    X_test_dl=X_test_embds,
    test_df=test_df,
    label_cols=label_cols,
    truth_csv='../test_labels.csv',
    models_dir='../models',
    reports_dir='../reports',
)